# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashidsami10000-afk/rashid-flyrank-internship-ml-owncopy/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook takes my Lane 4 capstone (CTR / Engagement Opportunity Scoring) and pins it down as a concrete ML task: what type of problem it is, what the target is, what "good" means, what one row stands for, and why a fixed rule isn't enough. Framing first, model later — every number below is produced by the code cells in this notebook.

## 1. My lane as an ML task: Scoring / Ranking

**Task type: Scoring (a priority score), used as a Ranking (an ordered review queue).**

The framing skill's table puts "which ones first?" in the **ranking / scoring** row — a priority score for every candidate, measured at the *top* of the list. That is exactly Lane 4:

| Question | Answer for Lane 4 |
|---|---|
| What decision? | Which visible pages get a review slot in the content queue, in what order |
| Who acts? | A FlyRank content analyst |
| What do they do? | Open the queue, read score + reason code, take one action: rewrite title/meta, improve intent match, or monitor |
| What is the output? | A **priority score per page**, ranked — not a yes/no verdict per page |

It is **not** a classification problem at the core: the analyst does not ask "this page has low CTR, yes/no" for all 22k pages individually — they ask "given I can only review ~50 pages, *which ones first*." That is precision@K over a ranking. (A binary `under_capturing` flag is only a convenient *evaluation* label, not the model's real output.)

The code below shows the shape of the task: how many pages are in the candidate pool, and how common "under-capturing" is — the **base rate** any ranking must beat.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane 4 slice: visible (enough impressions to matter) AND we know their position.
s = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

print(f"Candidate pool (visible + has position): {len(s)} of {len(df)} rows")
print("\nPosition tier mix of the pool:")
print(s["position_tier"].value_counts().to_string())

# Build the evaluation label this lane needs (built in section 2 properly).
vol = s[s["impressions_90d"] >= 1000]
tier_exp = vol.groupby("position_tier")["ctr"].median()
s["tier_expected_ctr"] = s["position_tier"].map(tier_exp)
s["gap_ratio"] = s["ctr"] / s["tier_expected_ctr"].where(s["tier_expected_ctr"] > 0)
s["under_capturing"] = (s["gap_ratio"] < 0.5).astype(int)

print(f"\nBase rate of under_capturing: {s['under_capturing'].mean():.3f}")
print("-> An UN-informed queue gets ~35% of its top-50 right just by random luck.")
print("   The task is to lift that well above base rate at the TOP of the queue.")


Candidate pool (visible + has position): 22006 of 30000 rows

Position tier mix of the pool:
position_tier
page_1      8633
page_3_5    6058
striking    5903
deep         879
top_3        533

Base rate of under_capturing: 0.353
-> An UN-informed queue gets ~35% of its top-50 right just by random luck.
   The task is to lift that well above base rate at the TOP of the queue.


## 2. Target or proxy

**The target is the CTR under-capture gap: how far below its position tier's expected CTR a page sits.**

The ideal target for this lane is *measured in the future*: "CTR that later changed after a fix." The starter dataset is a frozen 90-day snapshot with **no future window**, so for weeks 1–2 I use a **measured proxy** built from observed signals already in the data:

- `tier_expected_ctr` — the median CTR among pages in the same position tier with solid volume (impressions ≥ 1000), so a low-volume page doesn't distort the benchmark;
- `gap_ratio = actual_ctr / tier_expected_ctr` — 1.0 means "exactly average for its position", 0.5 means "half the CTR its position should earn";
- `under_capturing` (evaluation label) — `gap_ratio < 0.5`.

**Why this is a proxy, and kept honest:** the benchmark ("expected CTR for position X") is *computed from the observed data*, not handed down by a rule. But what I really care about is whether a page *will keep* under-capturing after a fix — that is a future-window label that lives in the warehouse release (`fact_content_daily_performance`), to be defined with the leakage rules in week 4+. Here I only *sketch* the label shape; the same column appears in section 4's dataframe.

**Label hygiene, from the skills:** I never use `trend_direction` / `trend_pct` (they are the declining-label source, and this lane is about clicks, not the decline bucket), and no product score is used as a feature.

In [2]:
# Build the target columns in one place (same code reused by the dataframe below).
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
s = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

vol = s[s["impressions_90d"] >= 1000]
tier_exp = vol.groupby("position_tier")["ctr"].median()
s["tier_expected_ctr"] = s["position_tier"].map(tier_exp)
s["gap_ratio"] = s["ctr"] / s["tier_expected_ctr"].where(s["tier_expected_ctr"] > 0)
s["under_capturing"] = (s["gap_ratio"] < 0.5).astype(int)

print("Expected CTR benchmark by position tier (median CTR, pages with >=1000 impressions):")
print(tier_exp.round(4).to_string())
print()
print("Distribution of the gap_ratio (1.0 = average for its position):")
print(s["gap_ratio"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(2).to_string())
print()
print("How many pages fall in the evaluation class (gap_ratio < 0.5):")
print(s["under_capturing"].value_counts().to_string())
print(f"Share under-capturing: {s['under_capturing'].mean():.3f}")


Expected CTR benchmark by position tier (median CTR, pages with >=1000 impressions):
position_tier
deep        0.00
page_1      0.24
page_3_5    0.10
striking    0.19
top_3       0.23

Distribution of the gap_ratio (1.0 = average for its position):
0.05    0.00
0.25    0.00
0.50    0.84
0.75    1.89
0.95    4.96

How many pages fall in the evaluation class (gap_ratio < 0.5):
under_capturing
0    14227
1     7779
Share under-capturing: 0.353


## 3. Success metric: Precision@K (K = 50)

**One defensible number: Precision@50 — of the top 50 pages the analyst actually reviews, what fraction really under-capture for their position.**

Why this one:
- It matches *how the output is used*: the analyst can act on a limited number of pages, so the top of the queue is the only place that matters. Average accuracy over 22k pages is the wrong lens (the guide's threshold section says exactly this).
- K = 50 is a realistic review capacity per batch; it can be tuned to the team's real capacity later (precision@20 for smaller teams).

**What I must beat (the honest yardsticks):**
- **Random / un-informed queue** → Precision@50 ≈ base rate (**0.35**).
- **A naive absolute rule** ("flag CTR < 0.2") → precision looks high only because it floods the top with `deep`/zero-CTR pages that nobody can fix at their position (shown in section 5) — a *cheap* high score that is useless in practice.
- A good Lane 4 ranking beats base rate **while** keeping the top-K populated with high-volume, realistic pages.

**A caution I will carry into the whole project:** the evaluation label (`gap_ratio < 0.5`) must never be recomputed from the model's own score. The model predicts the gap from *features known at decision time*; the label is a separate observed quantity. That separation is the leakage guard.

In [3]:
# Anchor the metric with real numbers from the data.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
s = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
vol = s[s["impressions_90d"] >= 1000]
tier_exp = vol.groupby("position_tier")["ctr"].median()
s["tier_expected_ctr"] = s["position_tier"].map(tier_exp)
s["gap_ratio"] = s["ctr"] / s["tier_expected_ctr"]
s["under_capturing"] = (s["gap_ratio"] < 0.5).astype(int)

base = s["under_capturing"].mean()
print(f"Base rate (random queue would get this Precision@50): {base:.3f}")

# Naive absolute rule that a non-analyst might write: "flag every page with CTR < 0.2"
rule = s[s["ctr"] < 0.2]
print(f"\nA naive 'CTR < 0.2' rule flags {len(rule)} of {len(s)} pages ({len(rule)/len(s):.0%})")
print("That is ~257x more pages than an analyst can review at K=50 ->")
print("a width-flag, not a ranking. The task is to ORDER these, not just flag them.")


Base rate (random queue would get this Precision@50): 0.353

A naive 'CTR < 0.2' rule flags 12847 of 22006 pages (58%)
That is ~257x more pages than an analyst can review at K=50 ->
a width-flag, not a ranking. The task is to ORDER these, not just flag them.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (one page) for one client, over the trailing 90-day window.**

The candidate pool is page-level, not day-level and not client-level: a *page* is the thing an analyst reviews and the thing the action (rewrite title/meta, improve intent match) is applied to. Client is used only for grouping / client-holdout splits, never as a feature.

The dataframe below shows exactly what one row means — including my target columns sketched on the right — so the label shape from section 2 is visible on real data.

In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
s = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
vol = s[s["impressions_90d"] >= 1000]
tier_exp = vol.groupby("position_tier")["ctr"].median()
s["tier_expected_ctr"] = s["position_tier"].map(tier_exp)
s["gap_ratio"] = s["ctr"] / s["tier_expected_ctr"]
s["under_capturing"] = (s["gap_ratio"] < 0.5).astype(int)

cols = [
    "content_id", "client_id",
    "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr",
    "tier_expected_ctr", "gap_ratio", "under_capturing",
]
unit = s.sort_values("gap_ratio")[cols]
print(f"Unit of analysis: one row = one content item (page), 90-day window. Pool = {len(unit)} pages.")
print("\nPreview, worst gap first (ctr is a x100 percentage: 0.35 = 0.35%):")
print(unit.head(6).round(4).to_string())
print("\nRow counts by position tier (the reviewable pool):")
print(s["position_tier"].value_counts().to_string())


Unit of analysis: one row = one content item (page), 90-day window. Pool = 22006 pages.

Preview, worst gap first (ctr is a x100 percentage: 0.35 = 0.35%):
                 content_id          client_id position_tier  avg_position  impressions_90d  clicks_90d  ctr  tier_expected_ctr  gap_ratio  under_capturing
29903  content_2aae951c32a6  client_349c41201b      striking          13.8              155           0  0.0               0.19        0.0                1
18696  content_05c9843a9494  client_4ec9599fc2        page_1           4.6              492           0  0.0               0.24        0.0                1
18699  content_fe70d7024017  client_349c41201b      page_3_5          21.8              968           0  0.0               0.10        0.0                1
18701  content_deb54e9e19cd  client_3fdba35f04      page_3_5          41.7             4560           0  0.0               0.10        0.0                1
18702  content_4ab57ee5efab  client_3fdba35f04      page_3_5    

## 5. Why ML beats a fixed rule here

**The pattern is too tangled for an if-statement because the *same* number means different things in different contexts.**

Three concrete failures of a fixed rule, all visible in the code below:

1. **A single CTR threshold is position-blind.** `ctr < 0.2` flags 12,847 pages (58% of the pool), yet 0.2% is *normal* for a `deep` page (median ~0) and a *red flag* for a `page_1` page (median 0.24). One threshold cannot be both.
2. **Even a tier-only benchmark is too coarse.** Within the same `page_1` position, mean CTR splits by content_type: feedly ~0.90% vs keyword ~0.35% vs comparison ~0.14% (verified in w01). So "expected for this position" is itself a function of format, intent, competition, age — many signals interacting, which is exactly when ML earns its place.
3. **The base rate is ~35% and the queue has hundreds of pages.** A rule can flag, but ordering 22k pages by "most fixable first" with many correlated inputs is the messy pattern ML is for.

**What ML has to earn:** a top-K that is *precision@K above base rate* AND *keeps high-volume realistic pages at the top* (an absolute-CTR shortcut loads the queue with deep-tier pages that can't be fixed at their position). If a transparent baseline (tier-median gap) already beats both, I keep the baseline — ML only wins this lane if it demonstrably beats the honest yardsticks in weeks ahead.

In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
s = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
vol = s[s["impressions_90d"] >= 1000]
tier_exp = vol.groupby("position_tier")["ctr"].median()
# deep-tier median CTR is 0 -> its pages have no reliable benchmark; exclude them from the gap.
s["tier_expected_ctr"] = s["position_tier"].map(tier_exp)
s["gap_ratio"] = None
s.loc[s["tier_expected_ctr"] > 0, "gap_ratio"] = (
    s.loc[s["tier_expected_ctr"] > 0, "ctr"]
    / s.loc[s["tier_expected_ctr"] > 0, "tier_expected_ctr"]
)
# Failure 1: one absolute threshold is position-blind.
rule = s[s["ctr"] < 0.2]
print(f"Failure 1: 'CTR < 0.2' flags {len(rule)} pages ({len(rule)/len(s):.0%} of the pool).")
print("  tier mix of what it flags:", dict(rule["position_tier"].value_counts(normalize=True).round(2)))

# Same absolute CTR, opposite meaning by tier:
deep_med = tier_exp.get("deep", 0)
p1_med = tier_exp["page_1"]
print(f"\n  0.2% vs its tier: deep median CTR = {deep_med}, page_1 median CTR = {p1_med}")
print("  -> a 0.2% page_1 page is at half its benchmark (fixable leak); a deep page is at ~normal.")

# Failure 2: tier-only benchmark misses content_type variation WITHIN a tier.
p1 = s[s["position_tier"] == "page_1"]
print(f"\nFailure 2: mean CTR within page_1 by content_type (tier ignored -> big miss):")
print(p1.groupby("content_type")["ctr"].mean().round(4).to_string())


# Preview the honest yardstick: gap-based ordering vs absolute-threshold ordering.
fin = s[s["gap_ratio"].notna()].copy()
print(f"\nYardstick preview: median impressions of gap_ratio-ranked top-50 vs naive top-50:"
      f" {int(fin.sort_values('gap_ratio').head(50).impressions_90d.median())} vs"
      f" {int(fin.sort_values('ctr').head(50).impressions_90d.median())}")


Failure 1: 'CTR < 0.2' flags 12847 pages (58% of the pool).
  tier mix of what it flags: {'page_3_5': np.float64(0.36), 'page_1': np.float64(0.3), 'striking': np.float64(0.26), 'deep': np.float64(0.06), 'top_3': np.float64(0.02)}

  0.2% vs its tier: deep median CTR = 0.0, page_1 median CTR = 0.24
  -> a 0.2% page_1 page is at half its benchmark (fixable leak); a deep page is at ~normal.

Failure 2: mean CTR within page_1 by content_type (tier ignored -> big miss):
content_type
comparison article    0.1412
feedly article        0.9048
keyword article       0.3458



Yardstick preview: median impressions of gap_ratio-ranked top-50 vs naive top-50: 388 vs 390


## Self-check

Before submitting, confirm each line honestly:

- [x] **Task type**: Scoring/Ranking (Lane 4) — named, with the decision, actor, action, and output table.
- [x] **Target/proxy**: CTR under-capture gap vs tier-expected CTR — a measured proxy on observed signals; future-window label deferred to the warehouse weeks. Built as real columns.
- [x] **Success metric**: Precision@K (K = 50), defended against base rate (0.35) and the naive absolute-threshold shortcut.
- [x] **Unit of analysis**: shown as a real dataframe — one row = one content page over the 90-day window, with target columns visible.
- [x] **Why ML not a rule**: three concrete rule failures demonstrated with real numbers (position-blind threshold, tier-only benchmark too coarse, ordering 22k pages).
- [x] Every section's claims are backed by an executed code cell in this notebook.
- [x] The notebook runs top to bottom with no errors (Run all).
- [x] No client names, URLs, or private queries anywhere; all IDs are pseudonyms.
- [x] Careful language throughout: observed, measured, directional, decision-support.
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
